[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C77_Video_World_Models_Course/01_video_latent/01_video_latent.ipynb)

# C77 模块 01 · 视频 latent 与 tokenizer

三件事：

1. **有效相关性 $\rho^{p_t}$** 给出选时间压缩率的上限；
2. **流式（因果）的代价非单调**，峰值在中间相关性，且集中在 chunk 的最后一帧；
3. **固定比特预算下「少而大的 token」更划算**（$1.39$–$1.58\times$）。

纯 numpy / CPU / 离线，不训练任何网络。

In [ ]:
import numpy as np

S_PIX, DPF = 8, 64          # 8x8 像素 = 64 维/帧

def spatial_basis(nmode=6):
    xs = np.arange(S_PIX)
    B = []
    for k in range(nmode):
        kx, ky = (k % 3) + 1, (k // 3) + 1
        B.append(np.outer(np.sin(np.pi * kx * (xs + 1) / (S_PIX + 1)),
                          np.cos(np.pi * ky * xs / S_PIX)).ravel())
    B = np.array(B)
    return B / np.linalg.norm(B, axis=1, keepdims=True)

BASIS = spatial_basis()

def make_seg(T=16, seed=0, corr=0.95, noise=0.05):
    r = np.random.default_rng(seed)
    nm = len(BASIS)
    a = np.zeros((T, nm))
    a[0] = r.normal(0, 1, nm)
    for t in range(1, T):
        a[t] = corr * a[t-1] + np.sqrt(1 - corr**2) * r.normal(0, 1, nm)
    return a @ BASIS + r.normal(0, noise, (T, DPF))

def gen(n, seed0, T=16, corr=0.95):
    return np.array([make_seg(T=T, seed=seed0+i, corr=corr) for i in range(n)])

def fit_pca(X, k):
    mu = X.mean(0, keepdims=True)
    _, _, Vt = np.linalg.svd(X - mu, full_matrices=False)
    return mu, Vt[:min(k, Vt.shape[0])]
def encode(X, mu, P):
    return (X - mu) @ P.T
def reg(Z, Y):
    return np.linalg.lstsq(np.column_stack([np.ones(len(Z)), Z]), Y, rcond=None)[0]
def apply_reg(Z, W):
    return np.column_stack([np.ones(len(Z)), Z]) @ W
def rel(Yh, Y):
    return float(np.sqrt(np.sum((Yh-Y)**2) / np.sum((Y - Y.mean(0, keepdims=True))**2)))

print('本模块只用 PCA（作为最优线性编码器）与 k-means（作为 VQ），无网络训练。')

## 1. 有效相关性 $\rho^{p_t}$

如果时间结构近似一阶马尔可夫，压缩 $p_t$ 倍后相邻 latent 帧的相关性是 $\rho^{p_t}$。

In [ ]:
print('  ρ（原始）  ' + '  '.join(f'p_t={p:<5d}' for p in (1,2,4,8,16)))
for rho in (0.99, 0.95, 0.90, 0.70):
    row = [rho**p for p in (1,2,4,8,16)]
    flag = ''
    below = [p for p, v in zip((1,2,4,8,16), row) if v < 0.3]
    if below:
        flag = f'   <- p_t>={below[0]} 时 ρ_eff<0.3'
    print(f'  {rho:9.2f}  ' + '  '.join(f'{v:<10.3f}' for v in row) + flag)

# 实测：latent 帧之间的相关性确实是 rho^pt
print()
print('实测验证（把每 p_t 帧的均值当 latent 帧，量它们的相邻相关性）:')
print('   ρ      p_t    理论 ρ^p_t   实测相邻相关   差')
for rho in (0.9, 0.95):
    for pt in (1, 2, 4, 8):
        segs = gen(3000, 0, T=64, corr=rho)
        nl = 64 // pt
        lat = segs.reshape(len(segs), nl, pt, DPF).mean(2)     # (n, nl, DPF)
        # 取第一个空间模式方向上的系数序列，量其相邻相关
        c = lat @ BASIS[0]
        a = c[:, :-1].ravel(); b = c[:, 1:].ravel()
        emp = float(np.corrcoef(a, b)[0,1])
        print(f'  {rho:.2f}   {pt:3d}    {rho**pt:10.4f}   {emp:12.4f}   {abs(rho**pt-emp):.4f}')
    print()
print('-> 时间平均会让实测值略高于 ρ^p_t（平均本身有平滑作用），')
print('   但趋势与量级一致，足以用来选 p_t 的上限。')

## 2. 流式（因果）的代价

总预算完全相同，差别只在解码第 $t$ 帧时能用哪些 latent。

In [ ]:
def causal_cost(T=16, s=4, k=8, corr=0.95, ntr=2500, nte=800, seed0=0):
    '''返回 (非因果逐帧误差, 因果逐帧误差)。总预算 = (T/s)*k，两者相同。'''
    TR, TE = gen(ntr, seed0, T=T, corr=corr), gen(nte, 900_000 + seed0, T=T, corr=corr)
    nch = T // s
    Ztr, Zte = [], []
    for j in range(nch):
        btr = TR[:, j*s:(j+1)*s, :].reshape(len(TR), -1)
        bte = TE[:, j*s:(j+1)*s, :].reshape(len(TE), -1)
        mu, P = fit_pca(btr, k)
        Ztr.append(encode(btr, mu, P)); Zte.append(encode(bte, mu, P))
    Ztr, Zte = np.concatenate(Ztr, 1), np.concatenate(Zte, 1)
    en, ec = [], []
    for t in range(T):
        Ytr, Yte = TR[:, t, :], TE[:, t, :]
        W = reg(Ztr, Ytr)
        en.append(rel(apply_reg(Zte, W), Yte))
        m = (t // s + 1) * k                       # 因果：只到 t//s 这个 chunk
        Wc = reg(Ztr[:, :m], Ytr)
        ec.append(rel(apply_reg(Zte[:, :m], Wc), Yte))
    return np.array(en), np.array(ec)

print('  s    k   总预算   非因果均值   因果均值   流式的代价')
for s, k in [(2,8), (4,16), (4,8), (8,16)]:
    en, ec = causal_cost(s=s, k=k)
    print(f'  {s:2d}  {k:3d}   {16//s*k:6d}   {en.mean():.5f}     {ec.mean():.5f}    '
          f'{ec.mean()/en.mean():.3f}x')
    assert ec.mean() >= en.mean() - 1e-9, '因果不可能优于非因果（信息更少）'
    assert ec.mean()/en.mean() < 1.05, f'代价应 <5%，得到 {ec.mean()/en.mean():.3f}'

print()
print('逐帧的代价分布（s=4, k=8）:')
en, ec = causal_cost(s=4, k=8)
ratios = ec / en
for j in range(4):
    seg = ratios[j*4:(j+1)*4]
    mark = '  <- 最后一个 chunk：因果与非因果等价' if j == 3 else ''
    print(f'  chunk {j}: ' + ' '.join(f'{v:.2f}' for v in seg) +
          f'   （峰值在第 {int(np.argmax(seg))} 帧）{mark}')

# 峰值在每个 chunk 的最后一帧
for j in range(3):
    seg = ratios[j*4:(j+1)*4]
    assert int(np.argmax(seg)) == 3, f'chunk {j} 的峰值应在最后一帧，得到 {np.argmax(seg)}'
# 最后一个 chunk 全为 1.00
assert np.allclose(ratios[12:], 1.0, atol=1e-6), '最后一个 chunk 应恒为 1.00'
print()
print('✅ 峰值落在每个 chunk 的**最后**一帧（那里最需要紧邻的未来）')
print('✅ 最后一个 chunk 恒为 1.00 —— 那里因果与非因果等价')

In [ ]:
# 代价对相关性是**非单调**的
print('代价 vs 时间相关性（s=4, k=8）:')
print('   corr    非因果误差   因果误差   流式的代价   解读')
costs = {}
for corr in (0.0, 0.3, 0.6, 0.8, 0.95, 0.995):
    en, ec = causal_cost(s=4, k=8, corr=corr)
    costs[corr] = ec.mean() / en.mean()
    note = '未来无信息' if corr < 0.1 else ('未来与过去高度冗余' if corr > 0.99 else '')
    print(f'  {corr:.3f}   {en.mean():.5f}    {ec.mean():.5f}   {costs[corr]:.4f}x     {note}')

_peak = max(costs, key=costs.get)
assert costs[0.0] < 1.005, f'corr=0 时代价应 ~1.00，得到 {costs[0.0]:.4f}'
assert costs[0.995] < costs[_peak], 'corr→1 时代价应回落'
assert _peak in (0.8, 0.95), f'峰值应在中间相关性，得到 {_peak}'
print()
print(f'✅ 峰值在 corr={_peak}（代价 {costs[_peak]:.4f}x），两端都回到 ~1.00')
print()
print('   两端都便宜的原因：corr=0 时未来什么也不提供；')
print('   corr→1 时未来与过去几乎是同一件事，看过去就等于看了未来。')
print('   只有中间 —— 未来带着过去没有的信息、而那信息确实有用 —— 才真的要付钱。')
print()
print('   工程含义：「因果 tokenizer 会明显掉质量」在本设定下不成立（最多 3.3%）。')
print('   它的真实成本在别处：**不能与图像 VAE 共享权重**，以及 chunk 边界的接缝。')

## 3. 固定比特预算：多而小 vs 少而大的 token

总比特 $= (T/p_t)\log_2 K$。固定它，比较不同的 $(p_t, K)$ 组合。

In [ ]:
def assign(X, C, chunk=4096):
    '''最近码字分配。用 ||x-c||² = ||x||² − 2x·c + ||c||² 走 matmul，
    避免建 (chunk, K, dim) 的中间数组（那个在 K 大时会到几十 GB）。'''
    cn = (C * C).sum(1)
    lab = np.empty(len(X), dtype=np.int64)
    for i in range(0, len(X), chunk):
        blk = X[i:i+chunk]
        d = cn[None, :] - 2.0 * (blk @ C.T)          # 省掉与 k 无关的 ||x||²
        lab[i:i+chunk] = np.argmin(d, 1)
    return lab

def kmeans(X, K, iters=30, seed=0):
    r = np.random.default_rng(seed)
    C = X[r.choice(len(X), K, replace=False)].copy()
    for _ in range(iters):
        lab = assign(X, C)
        cnt = np.bincount(lab, minlength=K)
        Cn = np.zeros_like(C)
        np.add.at(Cn, lab, X)
        nz = cnt > 0
        C[nz] = Cn[nz] / cnt[nz][:, None]
    return C

def vq_eval(Xtr, Xte, K, seed=0):
    '''返回 (重建相对误差, 码本利用率)。'''
    C = kmeans(Xtr, K, seed=seed)
    lab = assign(Xte, C)
    R = Xte - C[lab]
    err = float(np.sqrt(np.sum(R**2) / np.sum((Xte - Xte.mean(0))**2)))
    return err, len(np.unique(lab)) / K

T = 16
TR, TE = gen(4000, 0, T=T), gen(1000, 20_000, T=T)
print(f'一段视频 T={T} 帧 x {DPF} 维。总比特 = (T/p_t) * log2(K)')
print()
print('  比特预算   p_t   token 数   码本 K    重建相对误差   码本利用率')
best = {}
for bits in (32, 64):
    for pt in (1, 2, 4, 8):
        ntok = T // pt
        lg = bits / ntok
        if not (1 <= lg <= 10):
            continue
        K = int(round(2**lg))
        Xtr = TR.reshape(len(TR), ntok, pt*DPF).reshape(-1, pt*DPF)
        Xte = TE.reshape(len(TE), ntok, pt*DPF).reshape(-1, pt*DPF)
        if len(Xtr) < K:
            continue
        e, util = vq_eval(Xtr, Xte, K, seed=1)
        best.setdefault(bits, {})[pt] = (e, util, ntok, K)
        print(f'  {bits:8d}   {pt:3d}   {ntok:8d}   {K:6d}    {e:.5f}        {util*100:5.1f}%')
    print()

for bits in (32, 64):
    d = best[bits]
    pts = sorted(d)
    e_small, e_large = d[pts[0]][0], d[pts[-1]][0]
    assert e_large < e_small, f'{bits} 比特：少而大的 token 应更好'
    print(f'✅ {bits} 比特: p_t={pts[0]}（{d[pts[0]][2]} tok x {d[pts[0]][3]} 码本）'
          f'误差 {e_small:.5f}  vs  p_t={pts[-1]}（{d[pts[-1]][2]} tok x {d[pts[-1]][3]} 码本）'
          f'误差 {e_large:.5f}  ->  好 {e_small/e_large:.2f}x')
    assert all(v[1] > 0.99 for v in d.values()), '本规模下码本不该坍缩'
print()
print('✅ 全部配置的码本利用率 > 99% —— 本规模没有触到坍缩边界')
print()
print('-> 原因与第 1、2 节同源：**更长的时间跨度让相关性可以被利用**。')
print('   一个覆盖 4 帧的 token 只需编码「这 4 帧共同的运动」，')
print('   而 4 个各覆盖 1 帧的 token 必须各自重复那份共同信息。')

## ✏️ 练习 1：从数据估相邻帧相关性

实现 `estimate_rho(segs)`：给定一批视频段 `(n, T, D)`，
返回相邻帧相关性的**鲁棒估计**（用中位数而不是均值，
因为镜头边界会产生极端值）。

提示：对每段视频，算相邻帧向量的相关系数；然后对所有 (段, 位置) 取中位数。

In [ ]:
def estimate_rho(segs):
    '''相邻帧相关性的鲁棒估计。

    参数
    ----
    segs : (n, T, D) 视频批

    返回
    ----
    float : 全部 (段, 相邻位置) 的相关系数的中位数
    '''
    # TODO: 对每个 (i, t) 算 corr(segs[i,t], segs[i,t+1])（长度为 D 的两个向量），
    #       返回全部值的中位数。注意跳过方差为 0 的退化情形。
    raise NotImplementedError

In [ ]:
# 自测
print('   真 corr    估计的 ρ     误差')
for _c in (0.0, 0.3, 0.6, 0.9, 0.99):
    _segs = gen(400, 0, T=16, corr=_c)
    _rho = estimate_rho(_segs)
    print(f'  {_c:8.2f}   {_rho:9.4f}    {abs(_rho-_c):.4f}')
    assert abs(_rho - _c) < 0.15, f'真 corr={_c} 时估计 {_rho:.4f} 偏差过大'

# 鲁棒性：混入 20% 的「镜头切换」段（相邻帧完全无关）
print()
print('  鲁棒性检验：在 corr=0.9 的数据里混入 20% 的镜头切换段')
_good = gen(320, 0, T=16, corr=0.9)
_cut = gen(80, 500_000, T=16, corr=0.0)
_mixed = np.concatenate([_good, _cut], 0)
_rho_mixed = estimate_rho(_mixed)
_mean_based = float(np.mean([np.corrcoef(_mixed[i,t], _mixed[i,t+1])[0,1]
                             for i in range(len(_mixed)) for t in range(15)]))
print(f'    中位数估计 = {_rho_mixed:.4f}   （纯净数据是 {estimate_rho(_good):.4f}）')
print(f'    均值估计   = {_mean_based:.4f}   <- 被 20% 的切换段拉低了')
assert _rho_mixed > _mean_based + 0.05, '中位数应比均值更鲁棒'
assert abs(_rho_mixed - 0.9) < 0.12, '中位数应仍接近 0.9'

print()
print(f'✅ 中位数估计在 20% 污染下仍是 {_rho_mixed:.3f}，而均值被拉到 {_mean_based:.3f}')
print()
print('   工程含义：ρ 应该按**镜头**估而不是按数据集估。')
print('   混合估计会同时对慢镜头压得不够、对快剪辑压过头 ——')
print('   而中位数至少能告诉你「典型镜头」的 ρ 是多少。')

## ✏️ 练习 2：$p_t$ 的上限

实现 `max_pt(rho, T, rho_eff_min=0.3, max_occl=0, min_latent=None)`，
返回同时满足下面两个约束的最大时间压缩率：

- **收益约束**：$\rho^{p_t} \geq \rho_{\text{eff,min}}$；
- **上下文约束**：latent 帧数要够「跨过遮挡 + 再留 2 帧给位置和速度」，
  即 $T/p_t \geq \lceil \text{max\_occl}/p_t \rceil + 2$。
  化简后就是 $p_t \lesssim (T - \text{max\_occl})/2$——
  **可见跨度必须容得下至少 2 个 latent 帧**。

In [ ]:
def max_pt(rho, T, rho_eff_min=0.3, max_occl=0, min_latent=None,
           candidates=(1,2,4,8,16,32)):
    '''同时满足收益与上下文约束的最大 p_t。

    参数
    ----
    rho          : 原始相邻帧相关性
    T            : 帧数
    rho_eff_min  : 有效相关性的下限（低于它时时间压缩已无收益）
    max_occl     : 场景中最长的不可观测区间（原始帧）
    min_latent   : latent 帧数的硬下限（可选）
    candidates   : 候选 p_t

    返回
    ----
    (best_pt, reason) : 最大可行的 p_t，以及它被哪个约束卡住（'gain'/'context'/'none'）
    '''
    # TODO: 遍历 candidates（升序），对每个 pt 检查
    #       ① rho**pt >= rho_eff_min
    #       ② T//pt >= ceil(max_occl/pt) + 2，且 >= (min_latent or 1)
    #       返回最大可行的 pt，以及下一个候选被哪个条件卡住
    raise NotImplementedError

In [ ]:
# 自测
print('    ρ     T    最长遮挡   最大 p_t   卡在哪个约束')
_cases = [
    (0.99, 128, 0,  None),
    (0.95, 128, 0,  None),
    (0.90, 128, 0,  None),
    (0.70, 128, 0,  None),
    (0.99, 128, 24, None),
    (0.99, 32,  24, None),
    (0.99, 16,  24, None),
]
for _rho, _T, _occ, _ml in _cases:
    _pt, _why = max_pt(_rho, _T, max_occl=_occ, min_latent=_ml)
    print(f'  {_rho:.2f}  {_T:4d}   {_occ:8d}   {_pt:8d}   {_why}')

# ρ 越小，允许的 p_t 越小
_seq = [max_pt(r, 128)[0] for r in (0.99, 0.95, 0.90, 0.70)]
for _i in range(1, len(_seq)):
    assert _seq[_i] <= _seq[_i-1], f'ρ 越小 p_t 上限应越小：{_seq}'
assert max_pt(0.70, 128)[0] <= 4, 'ρ=0.70 时 p_t 上限应 <=4'
assert max_pt(0.99, 128)[0] >= 16, 'ρ=0.99 时 p_t 上限应 >=16'

# 短视频 + 长遮挡时被上下文卡住
_pt_s, _why_s = max_pt(0.99, 16, max_occl=24)
assert _why_s == 'context', f'T=16 + 遮挡 24 应被上下文卡住，得到 {_why_s}'
_pt_l, _why_l = max_pt(0.70, 128)
assert _why_l == 'gain', f'ρ=0.70 应被收益卡住，得到 {_why_l}'

print()
print(f'✅ ρ=0.99, T=128, 无遮挡: p_t 上限 {max_pt(0.99,128)[0]}（被收益约束卡住）')
print(f'✅ ρ=0.99, T=16, 遮挡 24 帧: p_t 上限 {_pt_s}（被**上下文**约束卡住）')
print(f'✅ ρ=0.70, T=128: p_t 上限 {_pt_l}（被收益约束卡住）')
print()
print('   两个约束会在不同场景下轮流成为瓶颈 ——')
print('   所以「业界都用 p_t=4」这种全局答案在两端都是错的。')

## ✏️ 练习 3：因果代价的峰值位置

实现 `causal_peak_frames(T, s, k, corr)`：返回因果/非因果误差比
在每个 chunk 内取最大值的**帧偏移**（相对 chunk 起点）。

正文说峰值在每个 chunk 的最后一帧。用这个函数在多个 $s$ 上验证。

In [ ]:
def causal_peak_frames(T=16, s=4, k=8, corr=0.95):
    '''每个 chunk 内因果代价最大的帧偏移。

    参数
    ----
    T, s, k, corr : 传给 causal_cost

    返回
    ----
    (offsets, ratios) :
      offsets = 每个 chunk 内 argmax(ratio) 的**帧内偏移**（0..s-1）；
                最后一个 chunk 因果与非因果等价，用 -1 表示
      ratios  = 完整的逐帧比值数组
    '''
    en, ec = causal_cost(T=T, s=s, k=k, corr=corr)
    ratios = ec / en
    # TODO: 把 ratios 切成 T//s 个 chunk；
    #       对每个 chunk 求 argmax 的帧内偏移；
    #       若该 chunk 的比值全都 ≈1.0（容差 1e-6），记 -1
    raise NotImplementedError

In [ ]:
# 自测
print('  (a) s >= 4：峰值稳定落在每个 chunk 的最后一帧')
print('    T   s   每个 chunk 内峰值的帧内偏移（-1 = 该 chunk 无代价）')
for _T, _s in [(16, 4), (16, 8), (24, 4), (24, 6)]:
    _off, _rat = causal_peak_frames(T=_T, s=_s)
    print(f'  {_T:4d}  {_s:2d}   {_off}')
    assert _off[-1] == -1, f'T={_T}, s={_s}: 最后一个 chunk 应无代价，得到 {_off[-1]}'
    for _j, _o in enumerate(_off[:-1]):
        assert _o == _s - 1, \
            f'T={_T}, s={_s}, chunk {_j}: 峰值应在偏移 {_s-1}，得到 {_o}'

print()
print('  (b) s = 2：chunk 内只有两帧，两者统计上无法区分')
_off2, _rat2 = causal_peak_frames(T=16, s=2)
_spread = []
for _j in range(7):                      # 跳过最后一个 chunk
    _seg = _rat2[_j*2:(_j+1)*2]
    _spread.append(float(abs(_seg[1] - _seg[0])))
print(f'    chunk 内两帧的比值之差: ' + ' '.join(f'{v:.2e}' for v in _spread))
print(f'    最大差 {max(_spread):.2e}  <-  远小于代价本身（~2%）')
assert max(_spread) < 5e-3, f's=2 时 chunk 内差应可忽略，得到 {max(_spread):.2e}'
print(f'    argmax 因此是噪声: {_off2[:7]}')

print()
print('  (c) 峰值的**显著性**（峰值 vs chunk 内最小值）:')
print('    s    峰值比值   chunk 内最小   峰/谷')
for _s in (4, 6, 8):
    _T = 24 if _s == 6 else 16
    _off, _rat = causal_peak_frames(T=_T, s=_s)
    _seg = _rat[:_s]
    print(f'   {_s:3d}    {_seg.max():.4f}    {_seg.min():.4f}      {_seg.max()/_seg.min():.4f}')
    assert _seg.max() / _seg.min() > 1.03, f's={_s} 时峰谷比应 >1.03'

print()
print('✅ s >= 4 时峰值稳定在每个 chunk 的**最后一帧**（偏移 s-1），峰谷比 > 1.03')
print('✅ s = 2 时 chunk 内差 < 5e-3 —— 结论的成立范围是 s >= 3')
print('✅ 最后一个 chunk 恒无代价（因果与非因果等价）')
print()
print('   工程含义：代价既小又**集中在 chunk 末尾**，所以可以用重叠 chunk 或')
print('   把 chunk 边界对齐到镜头切换来进一步摊薄 —— 而不需要放弃因果性。')
print('   而 s=2 那一行说明这个优化在很小的 chunk 上没有意义（本来就摊平了）。')

## ✏️ 练习 4：码本什么时候开始坍缩

正文说「少而大的 token 更划算」有一个前提：码本没坍缩。
实现 `codebook_collapse(pt, Ks, ntr)`：返回每个码本大小下的
**利用率**与**有效码本大小**（$2^{H}$，$H$ 为码字使用分布的熵，单位 bit）。

有效码本大小比利用率更有信息：利用率 100% 但分布极度不均时，
有效大小仍然远小于 $K$。

In [ ]:
def codebook_collapse(pt=4, Ks=(16, 64, 256, 1024), ntr=4000, nte=1000, T=16):
    '''每个码本大小下的利用率与有效码本大小。

    返回
    ----
    dict : K -> {'err', 'util', 'eff_K', 'entropy_bits'}
      util         = 被用到的码字比例
      entropy_bits = 码字使用分布的熵（bit）
      eff_K        = 2^entropy_bits（「有效码本大小」）
    '''
    tr, te = gen(ntr, 0, T=T), gen(nte, 20_000, T=T)
    ntok = T // pt
    Xtr = tr.reshape(len(tr), ntok, pt*DPF).reshape(-1, pt*DPF)
    Xte = te.reshape(len(te), ntok, pt*DPF).reshape(-1, pt*DPF)
    out = {}
    # TODO: 对每个 K：用 kmeans(Xtr, K, seed=1) 得码本；
    #       在 Xte 上分配码字（分块 argmin）；
    #       算 err（相对重建误差）、util、熵（用码字频率，跳过 0 频）、eff_K = 2^熵
    raise NotImplementedError

In [ ]:
# 自测
_res = codebook_collapse(pt=4, Ks=(16, 64, 256, 1024))
print('     K      重建误差   利用率    熵(bit)   有效码本 2^H   有效/K')
for _K in sorted(_res):
    _r = _res[_K]
    print(f'  {_K:6d}    {_r["err"]:.5f}   {_r["util"]*100:5.1f}%   '
          f'{_r["entropy_bits"]:6.2f}    {_r["eff_K"]:11.1f}   {_r["eff_K"]/_K:.3f}')

# 误差随 K 单调下降
_errs = [_res[K]['err'] for K in sorted(_res)]
for _i in range(1, len(_errs)):
    assert _errs[_i] < _errs[_i-1], f'误差应随 K 单调下降：{[round(e,5) for e in _errs]}'
# 有效/K 随 K 增大而下降（坍缩的定量形式）
_ratio = [_res[K]['eff_K']/K for K in sorted(_res)]
assert _ratio[-1] < _ratio[0], f'有效/K 应随 K 下降：{[round(r,3) for r in _ratio]}'
# 熵的增长慢于 log2(K)
_Ks = sorted(_res)
_H = [_res[K]['entropy_bits'] for K in _Ks]
_lg = [np.log2(K) for K in _Ks]
print()
print('  熵的增长 vs log2(K):')
for _K, _h, _l in zip(_Ks, _H, _lg):
    print(f'    K={_K:5d}: 熵 {_h:5.2f} bit   log2(K) = {_l:5.2f}   比值 {_h/_l:.3f}')
assert _H[-1]/_lg[-1] < _H[0]/_lg[0], '熵/log2(K) 应随 K 下降'

print()
print(f'✅ 误差随 K 单调下降（{_errs[0]:.5f} -> {_errs[-1]:.5f}）')
print(f'✅ 而**有效码本 / K** 从 {_ratio[0]:.3f} 降到 {_ratio[-1]:.3f} —— 这就是坍缩的定量形式')
print()
print('   注意利用率与有效大小的区别：利用率可以接近 100% 而分布极度不均，')
print(f'   此时有效码本远小于 K（本例 K={_Ks[-1]} 时有效只有 {_res[_Ks[-1]]["eff_K"]:.0f}）。')
print('   -> 所以「少而大的 token 更划算」这条结论必须带上「熵在增长」这个前提，')
print('      而检验它要看熵 / log2(K) 而不是利用率。')

## 📖 参考答案

In [ ]:
def estimate_rho(segs):
    '''相邻帧相关性的鲁棒估计（中位数）。'''
    vals = []
    n, T, D = segs.shape
    for i in range(n):
        for t in range(T - 1):
            a, b = segs[i, t], segs[i, t+1]
            sa, sb = a.std(), b.std()
            if sa < 1e-12 or sb < 1e-12:
                continue
            vals.append(float(np.corrcoef(a, b)[0, 1]))
    return float(np.median(vals))

def max_pt(rho, T, rho_eff_min=0.3, max_occl=0, min_latent=None,
           candidates=(1,2,4,8,16,32)):
    '''同时满足收益与上下文约束的最大 p_t。'''
    ok, why = 1, 'none'
    for pt in sorted(candidates):
        gain_ok = (rho ** pt) >= rho_eff_min
        need = max(int(np.ceil(max_occl / pt)) + 2, min_latent or 1)
        ctx_ok = (T // pt) >= need
        if gain_ok and ctx_ok:
            ok = pt
        else:
            why = 'gain' if not gain_ok else 'context'
            break
    return ok, why

def causal_peak_frames(T=16, s=4, k=8, corr=0.95):
    '''每个 chunk 内因果代价最大的帧偏移。'''
    en, ec = causal_cost(T=T, s=s, k=k, corr=corr)
    ratios = ec / en
    offs = []
    for j in range(T // s):
        seg = ratios[j*s:(j+1)*s]
        offs.append(-1 if np.allclose(seg, 1.0, atol=1e-6) else int(np.argmax(seg)))
    return offs, ratios

def codebook_collapse(pt=4, Ks=(16, 64, 256, 1024), ntr=4000, nte=1000, T=16):
    '''每个码本大小下的利用率与有效码本大小。'''
    tr, te = gen(ntr, 0, T=T), gen(nte, 20_000, T=T)
    ntok = T // pt
    Xtr = tr.reshape(len(tr), ntok, pt*DPF).reshape(-1, pt*DPF)
    Xte = te.reshape(len(te), ntok, pt*DPF).reshape(-1, pt*DPF)
    out = {}
    for K in Ks:
        if len(Xtr) < K:
            continue
        C = kmeans(Xtr, K, seed=1)
        lab = assign(Xte, C)
        R = Xte - C[lab]
        err = float(np.sqrt(np.sum(R**2) / np.sum((Xte - Xte.mean(0))**2)))
        cnt = np.bincount(lab, minlength=K).astype(float)
        p = cnt[cnt > 0] / cnt.sum()
        H = float(-np.sum(p * np.log2(p)))
        out[K] = {'err': err, 'util': float((cnt > 0).mean()),
                  'entropy_bits': H, 'eff_K': float(2.0**H)}
    return out

print('参考答案已定义。')
print()
print('要点：')
print('  1. ρ 要按**镜头**估、用**中位数** —— 混合估计在两端都错。')
print('  2. p_t 的上限由收益（ρ^p_t >= 0.3）与上下文（覆盖最长遮挡）**轮流**决定。')
print('  3. 因果代价既小（<=3.3%）又集中在 chunk 边界 —— 可以用重叠 chunk 摊薄。')
print('  4. 判断码本有没有坍缩要看**熵 / log2(K)**，不能看利用率。')

## 🧪 真实工程胶囊：一个 tokenizer 配置的验收清单

下面这段代码把本模块的四个量做成一次验收：给定一批素材，
它估出 $\rho$、给出 $p_t$ 的上限、量出因果代价、并检查码本的熵是否还在增长。

关键设计：**每一项都从数据算出来，没有一个是拍的**。
而输出不是「推荐配置」而是**一组上下界**，
因为这四项里有两项随 $p_t$ 改善、两项变差（见正文第 5 节的决策表）。

In [ ]:
def tokenizer_acceptance(segs, T, max_occl, bit_budget=None, pt_candidates=(1,2,4,8,16)):
    '''tokenizer 配置的验收清单。全部量都从 segs 估出。'''
    print('=' * 74)
    # ① 从数据估 ρ
    rho = estimate_rho(segs[:min(300, len(segs))])
    print(f'① 相邻帧相关性 ρ（中位数估计）= {rho:.4f}')

    # ② p_t 的上限
    pt_gain, why = max_pt(rho, T, max_occl=max_occl, candidates=pt_candidates)
    print(f'② p_t 上限 = {pt_gain}（被 {why} 约束卡住）')
    print(f'   逐档的有效相关性 ρ^p_t:')
    for pt in pt_candidates:
        need = int(np.ceil(max_occl / pt)) + 2
        have = T // pt
        ok = '✓' if (rho**pt >= 0.3 and have >= need) else '✗'
        print(f'     p_t={pt:3d}: ρ_eff={rho**pt:.4f}, latent 帧 {have} (需 >= {need})  {ok}')

    # ③ 因果代价（在选定的 p_t 上量）
    s = max(pt_gain, 2)
    en, ec = causal_cost(T=T if T <= 24 else 16, s=min(s, 8), k=8, corr=float(rho))
    cost = ec.mean() / en.mean()
    print(f'③ 流式（因果）代价 = {cost:.4f}x（s={min(s,8)}）')
    if cost > 1.10:
        print(f'   ⚠️  代价 > 10%：这个相关性区间对因果性最敏感')
    else:
        print(f'   ✓ 代价 <= 10%：因果性不需要用质量来论证')

    # ④ 码本的熵是否还在增长
    if bit_budget is not None:
        print(f'④ 固定 {bit_budget} 比特的配置对比:')
        rows = []
        for pt in pt_candidates:
            ntok = (T if T <= 16 else 16) // pt
            if ntok < 1:
                continue
            lg = bit_budget / ntok
            if not (1 <= lg <= 10):
                continue
            K = int(round(2**lg))
            r = codebook_collapse(pt=pt, Ks=(K,), ntr=1500, nte=400,
                                  T=(T if T <= 16 else 16))
            if K not in r:
                continue
            e = r[K]
            rows.append((pt, ntok, K, e['err'], e['entropy_bits']/np.log2(K)))
            print(f'     p_t={pt:3d}: {ntok:2d} tok x {K:5d} 码本 -> 误差 {e["err"]:.5f}, '
                  f'熵/log2(K) = {e["entropy_bits"]/np.log2(K):.3f}')
        if rows:
            best = min(rows, key=lambda x: x[3])
            print(f'   -> 该比特预算下最优是 p_t={best[0]}（误差 {best[3]:.5f}）')
            if best[4] < 0.7:
                print(f'   ⚠️  它的熵/log2(K) = {best[4]:.3f} < 0.7：码本已开始坍缩，'
                      f'「少而大更划算」这条结论在这里不再可靠')
    print('=' * 74)
    print()
    return dict(rho=rho, pt_max=pt_gain, bottleneck=why, causal_cost=float(cost))

print('=== 素材 A：慢镜头（ρ 高），长视频，无长遮挡 ===')
_A = tokenizer_acceptance(gen(400, 0, T=16, corr=0.97), T=128, max_occl=4, bit_budget=64)
assert _A['pt_max'] >= 8, 'ρ 高、无长遮挡时 p_t 上限应较大'

print('=== 素材 B：快剪辑（ρ 低）===')
_B = tokenizer_acceptance(gen(400, 0, T=16, corr=0.45), T=128, max_occl=4, bit_budget=64)
assert _B['pt_max'] < _A['pt_max'], 'ρ 低时 p_t 上限应更小'
assert _B['bottleneck'] == 'gain', 'ρ 低时应被收益约束卡住'

print('=== 素材 C：ρ 高但视频短、遮挡长 ===')
_C = tokenizer_acceptance(gen(400, 0, T=16, corr=0.97), T=16, max_occl=10, bit_budget=32)
assert _C['bottleneck'] == 'context', '短视频 + 长遮挡应被上下文约束卡住'

print('工程含义：')
print(f'  · 三份素材的 p_t 上限分别是 {_A["pt_max"]} / {_B["pt_max"]} / {_C["pt_max"]}，')
print(f'    而被卡住的约束分别是 {_A["bottleneck"]} / {_B["bottleneck"]} / {_C["bottleneck"]} ——')
print('    三种情形需要三种不同的应对，用一个全局 p_t 会在至少两处出错。')
print('  · 这份清单的输出是**上下界**而不是推荐值。')
print('    因为四项里两项随 p_t 改善（算力、比特效率）、两项变差（收益、上下文），')
print('    最终取哪个值是产品决策（要不要流式？要不要继承图像权重？），不是本清单能定的。')